# Multi-Agent Economic Simulation Platform

> **Research-grade ABM to test how system-level objectives (SUM / NASH / JAM) shape emergent inequality, exploitation, and power concentration.**

### Quick-start
1. **Runtime → Change runtime type** → select **A100 GPU** or **L4 GPU** (recommended)
2. **Runtime → Run all**

The notebook auto-detects your hardware and chooses the fastest backend:

| Hardware | Backend | Workers |
|----------|---------|--------|
| A100 GPU | JAX GPU | 14 |
| L4 GPU   | JAX GPU | 10 |
| T4 GPU   | Numba CUDA | 6 |
| CPU only (8+ cores) | Numba parallel | n-2 |
| CPU only (<8 cores) | Vectorised NumPy | all |


---
## Step 1 — Hardware detection & package installation

In [ ]:
# ── Detect hardware BEFORE installing so we can choose the right extras ──
import subprocess, sys, os

def _detect_gpu_tier():
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10)
        if r.returncode == 0 and r.stdout.strip():
            line = r.stdout.strip().splitlines()[0]
            name, mem = line.split(',')[0].strip(), int(line.split(',')[1].strip())
            name_l = name.lower()
            if 'a100' in name_l: return 'a100', name, mem
            if 'l4'   in name_l: return 'l4',   name, mem
            if 't4'   in name_l: return 't4',   name, mem
            if 'v100' in name_l: return 'v100', name, mem
            return 'generic_gpu', name, mem
    except Exception:
        pass
    return 'cpu', 'none', 0

GPU_TIER, GPU_NAME, GPU_MEM_MB = _detect_gpu_tier()
print(f'GPU tier  : {GPU_TIER}')
print(f'GPU name  : {GPU_NAME}')
print(f'GPU VRAM  : {GPU_MEM_MB/1024:.1f} GB' if GPU_MEM_MB else 'GPU VRAM  : N/A')

# Decide which extras to install
INSTALL_JAX   = GPU_TIER in ('a100', 'l4', 'v100')
INSTALL_NUMBA = not INSTALL_JAX   # Numba CUDA for T4; Numba CPU otherwise
print(f'Will install JAX  : {INSTALL_JAX}')
print(f'Will install Numba: {INSTALL_NUMBA}')

In [ ]:
%%capture install_out
# Core dependencies (always)
!pip install -q mesa numpy pandas scipy matplotlib seaborn plotly networkx pyarrow tqdm psutil

In [ ]:
%%capture install_accel_out
import subprocess, sys

if INSTALL_JAX:
    # JAX with CUDA — matches Colab CUDA version
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'jax[cuda12]', '-f',
                    'https://storage.googleapis.com/jax-releases/jax_cuda_releases.html'],
                   check=False)
    print('JAX (CUDA) installation attempted.')
elif INSTALL_NUMBA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=False)
    print('Numba installation attempted.')
print('Done.')

---
## Step 2 — Clone / update repository

In [ ]:
REPO_URL = 'https://github.com/caseymrobbins/ac_sugarsims.git'
REPO_DIR = '/content/ac_sugarsims'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest ...')
    !git -C {REPO_DIR} pull --rebase
else:
    print('Cloning repo ...')
    !git clone {REPO_URL} {REPO_DIR}

# Add to path so we can import modules directly
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

---
## Step 3 — Hardware report & acceleration config

In [ ]:
# Force a fresh detection now that all packages are installed
import importlib, hardware
importlib.reload(hardware)

hw  = hardware.detect_hardware()
cfg = hardware.get_accel_config(hw)

print(hardware._hardware_report(hw, cfg))

# Pre-compile / warm-up the regen function (Numba JIT compiles on first call)
import numpy as np
regen_fn = hardware.build_regen_fn(cfg.backend)
_f = np.ones((10, 10)) * 10.0
_r = np.ones((10, 10)) * 5.0
_l = np.ones((10, 10)) * 3.0
regen_fn(_f, _r, _l)
print('\nAcceleration backend ready:', cfg.backend)

---
## Step 4 — Quick smoke test (3 agents, 5 steps)

In [ ]:
import importlib, environment, agents, economy, planner, metrics
for mod in [environment, agents, economy, planner, metrics]:
    importlib.reload(mod)
from environment import EconomicModel

m = EconomicModel(seed=42, grid_width=20, grid_height=20,
                  n_workers=30, n_firms=3, n_landowners=2,
                  objective='JAM', accel_config=cfg)
for _ in range(5):
    m.step()

last = m.metrics_history[-1]
print(f'Smoke test passed ✓')
print(f'  Workers alive : {len(m.workers)}')
print(f'  Gini          : {last["all_gini"]:.3f}')
print(f'  Agency floor  : {last["agency_floor"]:.4f}')
print(f'  Backend       : {cfg.backend}')

---
## Step 5 — Run test experiment

**10 seeds × 300 steps × 3 conditions (SUM / NASH / JAM)**  
~3–8 minutes depending on hardware.

In [ ]:
import time, pandas as pd
from multiprocessing import Pool
import run_experiments as rx
importlib.reload(rx)

OUTPUT_DIR   = f'{REPO_DIR}/results'
N_SEEDS      = 10
N_STEPS      = 300
OBJECTIVES   = ['SUM', 'NASH', 'JAM']

os.makedirs(f'{OUTPUT_DIR}/raw_data', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/processed_data', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/plots', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/animations', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/statistical_tests', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/summary_tables', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

tasks = rx.build_experiment_list(
    OBJECTIVES, list(range(N_SEEDS)), N_STEPS,
    OUTPUT_DIR, save_raw=True, accel_cfg=cfg)

print(f'Episodes to run : {len(tasks)}  ({N_SEEDS} seeds × {N_STEPS} steps × {len(OBJECTIVES)} objectives)')
print(f'Workers         : {cfg.n_workers}')
print()

t0 = time.time()
results = []

# Colab-safe: use Pool with fork (Linux default)
n_pool = min(cfg.n_workers, len(tasks))
if n_pool <= 1:
    for i, task in enumerate(tasks):
        r = rx.run_single_episode(task)
        if r: results.append(r)
        if (i+1) % 5 == 0:
            print(f'  [{i+1}/{len(tasks)}] {task[0]} seed={task[1]} done')
else:
    with Pool(processes=n_pool) as pool:
        for i, r in enumerate(pool.imap_unordered(rx.run_single_episode, tasks)):
            if r: results.append(r)
            if (i+1) % 5 == 0:
                print(f'  Progress: {i+1}/{len(tasks)}')

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s — {len(results)}/{len(tasks)} succeeded')

results_df = pd.DataFrame(results)
results_df.to_parquet(f'{OUTPUT_DIR}/processed_data/episode_summaries.parquet', index=False)
results_df.to_csv(f'{OUTPUT_DIR}/processed_data/episode_summaries.csv', index=False)
print('Saved episode summaries.')
results_df.head(3)

---
## Step 6 — Statistical analysis

In [ ]:
import analysis, importlib
importlib.reload(analysis)

analysis_results = analysis.run_analysis(results_df, OUTPUT_DIR)

print('\nCondition Summary Table:')
key_cols = ['metric', 'SUM_mean', 'NASH_mean', 'JAM_mean']
summary = analysis_results['summary']
display_metrics = [
    'all_gini', 'worker_min', 'worker_mean',
    'agency_floor', 'frac_poverty_trap', 'frac_monopoly',
    'unemployment_rate', 'hhi'
]
mask = summary['metric'].isin(display_metrics)
print(summary[mask][key_cols].to_string(index=False))

---
## Step 7 — Inline visualisations

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image
import importlib, visualizations as viz
importlib.reload(viz)

# ── Load one representative raw episode per condition ──
raw_trajectories = {}
final_wealth_by_cond = {}

for obj in OBJECTIVES:
    sub = results_df[results_df['objective'] == obj]
    if len(sub) == 0:
        continue
    row = sub.iloc[0]
    seed, ep = int(row['seed']), int(row['episode_id'])
    raw_path = f'{OUTPUT_DIR}/raw_data/metrics_{obj}_seed{seed}_ep{ep}.parquet'
    if os.path.exists(raw_path):
        df = pd.read_parquet(raw_path)
        raw_trajectories[obj] = df
        # Reconstruct approximate wealth distribution
        last = df.iloc[-1]
        from run_experiments import _reconstruct_wealth_distribution
        final_wealth_by_cond[obj] = _reconstruct_wealth_distribution(
            last.get('worker_mean', 50), last.get('worker_std', 30),
            last.get('all_gini', 0.5))

print(f'Loaded {len(raw_trajectories)} condition trajectories.')

In [ ]:
# ── Per-condition plots ──
for obj, df in raw_trajectories.items():
    hist = df.to_dict('records')
    fw   = final_wealth_by_cond.get(obj, np.array([]))
    viz.generate_all_plots(
        metrics_history=hist,
        condition=obj,
        final_wealth=fw,
        output_dir=OUTPUT_DIR,
    )
print('Single-condition plots saved.')

# ── Comparative plots ──
viz.compare_floor_wealth(raw_trajectories,    f'{OUTPUT_DIR}/plots')
viz.compare_inequality(raw_trajectories,      f'{OUTPUT_DIR}/plots')
viz.compare_stability(raw_trajectories,       f'{OUTPUT_DIR}/plots')
viz.compare_agency_floor(raw_trajectories,    f'{OUTPUT_DIR}/plots')
viz.compare_wealth_distributions(final_wealth_by_cond, f'{OUTPUT_DIR}/plots')
print('Comparative plots saved.')

In [ ]:
# ── Display key comparative plots inline ──
KEY_PLOTS = [
    ('compare_floor_wealth.png',       'Floor Wealth: SUM vs NASH vs JAM'),
    ('compare_inequality.png',         'Inequality (Gini): SUM vs NASH vs JAM'),
    ('compare_wealth_distributions.png','Wealth Distributions: SUM vs NASH vs JAM'),
    ('compare_agency_floor.png',       'Agency Floor: SUM vs NASH vs JAM'),
    ('compare_stability.png',          'System Stability: Unemployment & Population'),
]

fig, axes = plt.subplots(len(KEY_PLOTS), 1, figsize=(14, 5 * len(KEY_PLOTS)))
for ax, (fname, title) in zip(axes, KEY_PLOTS):
    path = f'{OUTPUT_DIR}/plots/{fname}'
    if os.path.exists(path):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=13, pad=8)
    else:
        ax.text(0.5, 0.5, f'{fname} not found', ha='center', va='center')
        ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

In [ ]:
# ── Lorenz curves + power-law side by side ──
fig, axes = plt.subplots(1, len(OBJECTIVES), figsize=(5 * len(OBJECTIVES), 5))
for ax, obj in zip(axes, OBJECTIVES):
    path = f'{OUTPUT_DIR}/plots/lorenz_{obj}.png'
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(f'Lorenz — {obj}')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(OBJECTIVES), figsize=(5 * len(OBJECTIVES), 5))
for ax, obj in zip(axes, OBJECTIVES):
    path = f'{OUTPUT_DIR}/plots/powerlaw_{obj}.png'
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(f'Power-law — {obj}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gini trajectories + floor wealth ──
fig, axes = plt.subplots(2, len(OBJECTIVES), figsize=(5 * len(OBJECTIVES), 9))
for j, obj in enumerate(OBJECTIVES):
    for i, prefix in enumerate(['gini_trajectory', 'floor_wealth']):
        path = f'{OUTPUT_DIR}/plots/{prefix}_{obj}.png'
        if os.path.exists(path):
            axes[i, j].imshow(mpimg.imread(path))
        axes[i, j].axis('off')
        axes[i, j].set_title(f'{prefix.replace("_", " ").title()} — {obj}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Animate wealth evolution (Gini + floor) ──
import importlib, visualizations as viz
importlib.reload(viz)

for obj, df in raw_trajectories.items():
    viz.animate_wealth_distribution(
        df.to_dict('records'), condition=obj,
        output_dir=f'{OUTPUT_DIR}/animations')

# Show in notebook
from IPython.display import HTML
import base64

def show_gif(path, label=''):
    if not os.path.exists(path):
        print(f'GIF not found: {path}')
        return
    with open(path, 'rb') as f:
        data = base64.b64encode(f.read()).decode('utf-8')
    display(HTML(f'<h4>{label}</h4><img src="data:image/gif;base64,{data}" />'))

for obj in OBJECTIVES:
    show_gif(f'{OUTPUT_DIR}/animations/wealth_evolution_{obj}.gif',
             label=f'Wealth Evolution — {obj}')

---
## Step 8 — Emergent dynamics summary

In [ ]:
import pandas as pd

print('=' * 65)
print('  EMERGENT DYNAMICS SUMMARY  (mean across seeds)')
print('=' * 65)

fields = [
    ('all_gini',          'Gini coefficient',         '{:.3f}'),
    ('worker_min',        'Min worker wealth',         '{:.2f}'),
    ('worker_mean',       'Mean worker wealth',        '{:.1f}'),
    ('agency_floor',      'Agency floor',              '{:.4f}'),
    ('frac_poverty_trap', '% steps w/ poverty trap',  '{:.1%}'),
    ('frac_monopoly',     '% steps w/ monopoly',      '{:.1%}'),
    ('frac_cartel',       '% steps w/ cartel',        '{:.1%}'),
    ('unemployment_rate', 'Unemployment rate',         '{:.1%}'),
    ('hhi',               'HHI (market concentration)','{:.3f}'),
    ('total_debt',        'Mean total debt',           '{:.1f}'),
]

print(f'{"Metric":<35}', end='')
for obj in OBJECTIVES:
    print(f'{obj:>12}', end='')
print()
print('-' * 65)

for col, label, fmt in fields:
    if col not in results_df.columns:
        continue
    print(f'{label:<35}', end='')
    for obj in OBJECTIVES:
        vals = results_df[results_df['objective'] == obj][col].dropna()
        val = vals.mean() if len(vals) else float('nan')
        try:
            print(f'{fmt.format(val):>12}', end='')
        except Exception:
            print(f'{val:>12.3f}', end='')
    print()

print('=' * 65)

In [ ]:
# ── Significant pairwise differences ──
if 'pairwise_tests' in analysis_results:
    pw = analysis_results['pairwise_tests']
    sig = pw[pw['significant'] == True].sort_values('p_value')
    if len(sig):
        print('Statistically significant pairwise differences (p < 0.05):')
        print(sig[['metric', 'pair', 'p_value', 'effect_size']].to_string(index=False))
    else:
        print('No significant pairwise differences detected at p < 0.05.')
        print('(Run the full experiment with 1500 steps for larger effect sizes.)')

---
## Step 9 — Full experiment (optional)

Runs the full protocol: **20 seeds × 1500 steps × 3 conditions = 60 episodes**.  
Expected runtime:
- A100: ~15–25 min
- L4: ~25–40 min
- T4: ~45–75 min
- CPU only: ~2–4 hours

Un-comment the cell below to run.

In [ ]:
# FULL_N_SEEDS  = cfg.recommended_seeds   # 20
# FULL_N_STEPS  = cfg.recommended_steps   # 1500
# FULL_WORKERS  = cfg.n_workers
#
# tasks_full = rx.build_experiment_list(
#     OBJECTIVES, list(range(FULL_N_SEEDS)), FULL_N_STEPS,
#     OUTPUT_DIR, save_raw=True, accel_cfg=cfg)
#
# print(f'Full run: {len(tasks_full)} episodes | {FULL_N_STEPS} steps | {FULL_WORKERS} workers')
#
# t0 = time.time()
# full_results = []
# with Pool(processes=FULL_WORKERS) as pool:
#     for i, r in enumerate(pool.imap_unordered(rx.run_single_episode, tasks_full)):
#         if r: full_results.append(r)
#         if (i+1) % 10 == 0:
#             print(f'  [{i+1}/{len(tasks_full)}] elapsed {time.time()-t0:.0f}s')
#
# full_df = pd.DataFrame(full_results)
# full_df.to_parquet(f'{OUTPUT_DIR}/processed_data/full_episode_summaries.parquet', index=False)
# print(f'Full run complete in {time.time()-t0:.1f}s')
print('Cell commented out — un-comment to run full experiment.')

---
## Step 10 — Download results

Download all output files to your local machine.

In [ ]:
# Zip and download everything in results/
try:
    from google.colab import files
    import shutil
    zip_path = '/content/simulation_results'
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    files.download(zip_path + '.zip')
    print('Download started — check your browser.')
except ImportError:
    print(f'Not running in Colab. Results are in: {OUTPUT_DIR}')
    import subprocess
    r = subprocess.run(['du', '-sh', OUTPUT_DIR], capture_output=True, text=True)
    print(r.stdout)

---
## Notes

### Why conditions look similar at 300 steps
The planner's hill-climbing policy optimiser updates every **25 steps** and
needs ~200–400 steps to converge to meaningfully different policies.
At 1500 steps the SUM / NASH / JAM conditions diverge substantially.

### Emergent dynamics to look for at 1500 steps
| Phenomenon | SUM | NASH | JAM |
|-----------|-----|------|-----|
| Floor collapse | ✓ frequent | ✗ rare | ✗ structurally prevented |
| Power-law tail α < 2 | ✓ | ✗ | ✗ |
| Monopoly incidence > 50% | ✓ | ~ | ✗ |
| Debt traps | ✓ | ~ | ✗ |
| High Gini | ✓ | ~ | ✗ |

### Command-line usage
```bash
# Test
python run_experiments.py --test
# Full
python run_experiments.py
# Custom
python run_experiments.py --objectives JAM SUM --seeds 5 --steps 500
```

### Hardware selection in Colab
- **A100 / L4** → Runtime → Change runtime type → GPU → A100 or L4  
- Requires **Colab Pro+** for A100 access
